<a href="https://colab.research.google.com/github/eojin22/ESAA/blob/main/OB_WEEK03_1.%EC%88%98%EC%83%81%EC%9E%91%EB%A6%AC%EB%B7%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **0918 수상작 리뷰**



# 주제 및 데이터

### **주제**: **주차수요 예측 AI 경진대회**

### **데이터 구성:**

* **train.csv**

  * `단지코드` : 임대주택 단지별 코드
  * `총세대수` : 단지 내 전체 세대 수
  * `임대건물구분`, `지역`, `공급유형` : 주택 및 지역 관련 정보
  * `전용면적`, `전용면적별세대수` : 주택 규모와 세대 수 정보
  * `공가수` : 비어 있는 세대 수
  * `신분` : 임대주택 입주자의 신분 관련 정보
  * `임대료보증금`, `임대료` : 임대 조건 관련 정보
  * `도보 10분거리 내 지하철역 수`, `도보 10분거리 내 버스정류장 수` : 대중교통 접근성 정보
  * `단지내주차면수` : 단지 내 주차 가능 면수
  * `등록차량수` : 예측 대상인 등록 차량 수

* **test.csv**

  * train 데이터와 동일한 설명변수를 포함
  * `등록차량수`가 제공되지 않으며 이를 예측하는 것이 목표

* **age_gender_info.csv**

  * 지역별 연령 및 성별 인구 분포 데이터
  * 10대 미만부터 100대까지의 연령대별 남녀 인구 정보를 포함





---



## **코드 리뷰**

### 1. 데이터 확인 및 전처리

* 데이터의 자료형과 결측치 등을 확인하며 기본적인 데이터 구조를 파악
* 지하철역 수와 버스정류장 수의 결측값을 **0으로 대체**
* 주차수요와 관련성이 높다고 판단되는 변수들을 활용하여 파생변수를 생성

### 2. Feature Engineering

* **지역 변수 통합**

  * 등록차량수의 중앙값을 기준으로 지역별 주차수요 특성을 확인하고, 유사한 특성을 가진 지역을 5개의 그룹으로 통합

* **공급유형 변수 통합**

  * `등록차량수 - 단지내주차면수`의 차이를 기준으로 공급유형을 5개의 그룹으로 재구성
  * 단순히 기존 범주를 사용하는 것이 아니라 주차수요와 관련된 정보를 기준으로 새로운 범주를 생성

* **전용면적 변환**

  * 전용면적을 5단위로 변환하여 비슷한 규모의 주택을 하나의 구간으로 묶음

### 3. 모델링

#### 1) 단순회귀

* 먼저 선형회귀를 활용하여 주차수요와 각 변수의 관계를 파악
* `StandardScaler`를 활용하여 변수의 스케일을 표준화
* **Permutation Importance**를 활용하여 각 변수의 중요도를 확인하고 최종적으로 사용할 변수를 선정

#### 2) 다항회귀

* 다중공산성과 회귀분석 결과, Permutation Importance 등을 활용하여 최종 피처를 선정
* 이후 `PolynomialFeatures`를 사용하여 기존 변수들의 2차항과 상호작용항을 생성
* 선형적인 관계만을 가정하는 단순회귀보다 변수 간의 비선형적인 관계를 반영하여 주차수요를 예측

### 4. 모형 개선

#### 1) Cook's Distance를 활용한 가중치 생성

* 각 관측값이 회귀모형에 미치는 영향력을 확인하기 위해 **Cook's Distance**를 활용
* 영향력이 큰 관측값과 작은 관측값을 구분한 뒤, Cook's Distance가 작은 관측값에 상대적으로 큰 가중치를 부여
* 이를 통해 일부 영향력 있는 관측값에 의해 회귀모형이 크게 영향을 받는 문제를 완화하고자 함

#### 2) Stratified K-Fold

* 데이터의 특성을 고려하여 **Stratified K-Fold**를 적용
* 단순히 한 번의 학습/검증으로 모델을 평가하는 것보다 여러 번의 분할을 통해 모델의 성능을 확인하고자 함



In [ ]:
# 로그 변환
train['log_임대료보증금'] = np.log1p(train['임대료보증금'])
train['log_임대료'] = np.log1p(train['임대료'])

test['log_임대료보증금'] = np.log1p(test['임대료보증금'])
test['log_임대료'] = np.log1p(test['임대료'])

In [ ]:
# Linear Regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

X_train, X_valid, y_train, y_valid = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

model = LinearRegression()

model.fit(X_train, y_train)

pred = model.predict(X_valid)

mean_absolute_error(y_valid, pred)

In [ ]:
# 다항회귀
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(
    degree=2,
    include_bias=False
)

X_poly = poly.fit_transform(X_scaled)



---



## **차별점 및 배울 점**

**1. EDA를 바탕으로 한 파생변수 생성**

* 단순히 결측치를 처리하고 데이터를 확인하는 것에 그치지 않고, **주차수요와 변수 간의 관계를 분석한 결과를 바탕으로 새로운 변수를 생성**했다는 점이 인상적이었다.
* 특히 지역과 공급유형을 기존 범주 그대로 사용하지 않고 등록차량수와 단지 내 주차면수의 관계를 이용해 재구성한 점에서, 데이터의 특성을 파악한 후 전처리와 Feature Engineering을 진행하는 것이 중요하다는 것을 알 수 있었다.

**2. 다항회귀를 활용한 변수 간 관계 반영**

* 일반적인 선형회귀에서 끝내지 않고 `PolynomialFeatures`를 활용하여 변수 간의 2차항과 상호작용항을 생성하였다.
* 이를 통해 주차수요가 하나의 변수에 의해 단순하게 결정되는 것이 아니라 여러 변수의 조합에 따라 달라질 수 있다는 점을 모델에 반영하려고 한 점이 인상적이었다.

**3. Cook's Distance를 활용한 회귀모형 개선**

* 단순히 모델을 선택하거나 하이퍼파라미터를 조정하는 방식이 아니라, **각 관측값이 모델에 미치는 영향력 자체를 분석하여 가중치를 부여**했다는 점이 차별적이었다.
* 이를 통해 회귀분석에서 이상치나 영향력 있는 관측값이 모델에 미치는 영향을 확인하는 방법을 실제 예측 문제에 적용해 볼 수 있었다.

